# Notebook 8 . Bandingkan Dua CSV

Bandingkan dua versi data berdasarkan kolom kunci. Notebook melaporkan baris baru, baris yang hilang, dan baris dengan isi yang berubah. Kolom kunci harus unik pada masing-masing file agar perbandingan tidak ambigu.

In [ ]:
from pathlib import Path
import pandas as pd

# PENGATURAN YANG BISA DIUBAH
CSV_Lama = Path("Input/sebelum.csv")
CSV_Baru = Path("Input/sesudah.csv")
Kolom_Kunci = ["id"]
Separator_CSV = ","  # Ganti menjadi ";" jika CSV memakai titik koma
Folder_Output = Path("Output")

def find_project_dir():
    current = Path.cwd().resolve()
    for folder in (current, *current.parents):
        if (folder / "Input").is_dir():
            (folder / "Output").mkdir(parents=True, exist_ok=True)
            return folder
    return current

BASE_DIR = find_project_dir()


## Hitung perbedaan

In [ ]:
def read_csv(path):
    path = path if path.is_absolute() else BASE_DIR / path
    if not path.exists(): raise FileNotFoundError(f"File tidak ditemukan: {path}")
    frame = pd.read_csv(path, sep=Separator_CSV, dtype="string", encoding="utf-8-sig")
    frame.columns = frame.columns.astype(str).str.strip()
    return frame
old, new = read_csv(CSV_Lama), read_csv(CSV_Baru)
for label, frame in [("sebelum", old), ("sesudah", new)]:
    missing = [c for c in Kolom_Kunci if c not in frame]
    if missing: raise KeyError(f"Kolom kunci tidak ditemukan di file {label}: {missing}")
    if frame.duplicated(Kolom_Kunci).any(): raise ValueError(f"Kolom kunci tidak unik di file {label}; rapikan duplikat dahulu.")
old_idx = old.set_index(Kolom_Kunci, drop=False)
new_idx = new.set_index(Kolom_Kunci, drop=False)
old_keys, new_keys = old_idx.index, new_idx.index
added_keys = new_keys.difference(old_keys)
removed_keys = old_keys.difference(new_keys)
common = old_keys.intersection(new_keys)
compare_cols = [c for c in old.columns if c not in Kolom_Kunci and c in new.columns]
changed_mask = pd.Series(False, index=common)
for col in compare_cols:
    changed_mask |= old_idx.loc[common, col].fillna("<NA>").ne(new_idx.loc[common, col].fillna("<NA>"))
changed_keys = common[changed_mask.to_numpy()]
added = new_idx.loc[added_keys].reset_index(drop=True)
removed = old_idx.loc[removed_keys].reset_index(drop=True)
changed = new_idx.loc[changed_keys].reset_index(drop=True)
print(f"Baris baru: {len(added):,}; hilang: {len(removed):,}; berubah: {len(changed):,}")
display(changed.head())


In [ ]:
outdir = Folder_Output if Folder_Output.is_absolute() else BASE_DIR / Folder_Output
outdir.mkdir(parents=True, exist_ok=True)
for name, frame in [("baris_baru.csv", added), ("baris_hilang.csv", removed), ("baris_berubah_versi_baru.csv", changed)]:
    frame.to_csv(outdir / name, index=False, encoding="utf-8-sig", sep=Separator_CSV)
print(f"Laporan tersimpan di {outdir.resolve()}")